In [4]:
import os
import json
from collections import defaultdict


In [5]:
#scores = ["proto_margin", "similarity", "soft_knn_margin_all", "soft_knn_margin_topk"]
scores = ["soft_knn_margin_all", "soft_knn_margin_topk"]
IS_ZOO = True  # Whether we are working with the Zoo dataset or not
model_variants = ["base", "ViTG"]
output_dir = "/sc/home/iven.schlegelmilch/bachelor_thesis_code/visualizations"

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# --- Step 1: Group files by score type ---
# We use a dictionary to store the file paths for each score.
# e.g., {"proto_margin": {"base": "path/to/base_file.json", "ViTG": "path/to/vitg_file.json"}}
file_pairs = defaultdict(dict)

print("Scanning directory and grouping files...")

for filename in os.listdir(output_dir):
    # Filter out non-json files and our own output files
    if not IS_ZOO and not filename.endswith(".json") or "PAIRED" in filename or "renamed_body" not in filename:
        continue
    if IS_ZOO and "renamed_body_subsampled_20pct" not in filename:
        continue
    # Identify which score and variant this file belongs to
    for score in scores:
        if score in filename:
            for variant in model_variants:
                if variant in filename:
                    # We found a match! Store the full path.
                    file_pairs[score][variant] = os.path.join(output_dir, filename)
                    print(f"  - Found '{variant}' file for '{score}': {filename}")
                    # Break inner loops once a file is categorized
                    break
            break

    

Scanning directory and grouping files...
  - Found 'ViTG' file for 'soft_knn_margin_all': ViTG-body_face-spac23+24v3-dedup_bp_transforms_squared_renamed_body_subsampled_20pct_soft_knn_margin_all_conv_gamma_0p75_lin_gamma_0p001_proxy_temp_0p005_db.pt.json
  - Found 'ViTG' file for 'soft_knn_margin_topk': ViTG-body_face-spac23+24v3-dedup_bp_transforms_squared_renamed_body_subsampled_20pct_soft_knn_margin_topk_conv_gamma_0p75_lin_gamma_0p0001_proxy_temp_0p005_topk_7500_db.pt.json
  - Found 'base' file for 'soft_knn_margin_all': base_bp_transforms_squared_renamed_body_subsampled_20pct_soft_knn_margin_all_conv_gamma_0p05_lin_gamma_0p01_proxy_temp_0p005_db.pt.json
  - Found 'base' file for 'soft_knn_margin_topk': base_bp_transforms_squared_renamed_body_subsampled_20pct_soft_knn_margin_topk_conv_gamma_0p25_lin_gamma_0p01_proxy_temp_0p005_topk_1500_db.pt.json


In [6]:
for score, paths in file_pairs.items():
    # Check if we have a complete pair (both 'base' and 'ViTG')
    if "base" not in paths or "ViTG" not in paths:
        print(f"  - WARNING: Skipping '{score}'. Found only one variant, not a complete pair.")
        if "base" in paths: print(f"    - Found: {paths['base']}")
        if "ViTG" in paths: print(f"    - Found: {paths['ViTG']}")
        continue

    base_filepath = paths["base"]
    vitg_filepath = paths["ViTG"]

    # Load the data from both JSON files
    try:
        with open(base_filepath, 'r') as f:
            base_data = json.load(f)
        with open(vitg_filepath, 'r') as f:
            vitg_data = json.load(f)
        print(f"\nProcessing pair for '{score}'...")
        print(f"  - Base file: {os.path.basename(base_filepath)}")
        print(f"  - ViTG file: {os.path.basename(vitg_filepath)}")
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"\nERROR: Could not read or parse files for '{score}'. Skipping. Error: {e}")
        continue

    # Compute the intersection for each list category
    intersection_results = {}
    common_keys = set(base_data.keys()) & set(vitg_data.keys())

    for key in common_keys:
        # Get the lists, defaulting to an empty list if a key is somehow missing
        base_ids = base_data.get(key, [])
        vitg_ids = vitg_data.get(key, [])

        # Use sets for efficient intersection
        intersecting_ids = set(base_ids).intersection(set(vitg_ids))

        # Convert back to a sorted list for consistent output
        intersection_results[key] = sorted(list(intersecting_ids))
        
        print(f"    - Found {len(intersection_results[key])} intersecting items for '{key}' out of {len(base_ids)} (base) and {len(vitg_ids)} (ViTG)")

    # Save the intersection results to a new JSON file
    if IS_ZOO:
        output_filename = f"{score}_PAIRED_intersection_zoo.json"
    else:
        output_filename = f"{score}_PAIRED_intersection.json"

    output_filepath = os.path.join(output_dir, output_filename)

    with open(output_filepath, 'w') as f:
        json.dump(intersection_results, f, indent=4)

    print(f"  -> Saved intersection results to: {output_filepath}")

print("\nIntersection process complete.")


Processing pair for 'soft_knn_margin_all'...
  - Base file: base_bp_transforms_squared_renamed_body_subsampled_20pct_soft_knn_margin_all_conv_gamma_0p05_lin_gamma_0p01_proxy_temp_0p005_db.pt.json
  - ViTG file: ViTG-body_face-spac23+24v3-dedup_bp_transforms_squared_renamed_body_subsampled_20pct_soft_knn_margin_all_conv_gamma_0p75_lin_gamma_0p001_proxy_temp_0p005_db.pt.json
    - Found 104 intersecting items for 'positive_lerf_flippers' out of 196 (base) and 170 (ViTG)
    - Found 18196 intersecting items for 'robust_morf_successes' out of 27932 (base) and 29470 (ViTG)
    - Found 9332 intersecting items for 'hard_lerf_failures' out of 15751 (base) and 14332 (ViTG)
    - Found 247 intersecting items for 'negative_morf_flippers' out of 424 (base) and 464 (ViTG)
  -> Saved intersection results to: /sc/home/iven.schlegelmilch/bachelor_thesis_code/visualizations/soft_knn_margin_all_PAIRED_intersection_zoo.json

Processing pair for 'soft_knn_margin_topk'...
  - Base file: base_bp_transforms